In [1]:
import pandas as pd
import os
# from transformers import BertTokenizer, BertModel
from bert_score import BERTScorer
import re
import pickle
import nltk
from nltk.translate import meteor
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob
import math

/Users/isabel/anaconda3/envs/entityRecognitionNotes/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /Users/isabel/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     /Users/isabel/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /Users/isabel/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /Users/isabel/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /Users/isabel/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_ru is already
[nltk_data]    |       up-to-date!
[nlt

True

In [3]:
gpt3_load = './generatedDataFromGoogleDrive_gpt3/data/'
gpt4_load = './generatedDataFromGoogleDrive_gpt4/data/'
gpt3_save_bert = './metrics/gpt3_F1_BERT_scores.pkl'
gpt4_save_bert = './metrics/gpt4_F1_BERT_scores.pkl'
gpt3_save_meteor = './metrics/gpt3_meteor_scores.pkl'
gpt4_save_meteor = './metrics/gpt4_meteor_scores.pkl'
gpt3_save_sentiment = './metrics/gpt3_sentiment_scores.pkl'
gpt4_save_sentiment = './metrics/gpt4_sentiment_scores.pkl'

# Contextual Similarity - BERTScore

In [6]:
def bertScore(loadingFile, savingFile):
    # load all files from data folder (generated in Google Colab)
    met_dfs = {}
    unmet_dfs = {}
    for file in os.listdir(loadingFile):
        if '.csv' in file:
            
            if 'unmet' in file:
                unmet_dfs[file.replace('.csv', '')] = pd.read_csv(f'{loadingFile}{file}')
            else:
                met_dfs[file.replace('.csv', '')] = pd.read_csv(f'{loadingFile}{file}')
    nurse_notes = pd.read_excel('../../fake_notes.xlsx')

    score_dictionary = {}
    scorer = BERTScorer(model_type="bert-base-uncased")
    for i in range(len(nurse_notes)):
        word = ""
        # BERTScore penalizes punctuation, we should take this into account - https://aclanthology.org/2023.findings-acl.381.pdf
        if i < 5:
            candidate =  [re.sub(r'[^\w\s/]', '', i) for i in met_dfs[f'{i % (len(nurse_notes) // 2)}_met']['report']]
            word = "Met"
        else:
            candidate =  [re.sub(r'[^\w\s/]', '', i) for i in unmet_dfs[f'{i % (len(nurse_notes) // 2)}_unmet']['report']]
            word = "Unmet"
        reference = [re.sub(r'[^\w\s/]', '', nurse_notes['Note'][i])] * (len(candidate))
        try:
            P, R, F1 = scorer.score(candidate, reference)
        except Exception as e:
            print(e)
        # save the F1 values, as these are recommended for use by the BERTScore authors - http://arxiv.org/abs/1904.09675
        score_dictionary[f'{i}_{word.lower()}'] = F1.mean()
        print(f"{word} {i} F1 Score: {F1.mean()}")

    # put scores in a pickle file
    with open(savingFile, 'wb') as f:
        pickle.dump(score_dictionary, f)
    

In [7]:
bertScore(gpt4_load, gpt4_save_bert)

Met 0 F1 Score: 0.6469533443450928
Met 1 F1 Score: 0.6959341168403625
Met 2 F1 Score: 0.6237338781356812
Met 3 F1 Score: 0.6774038076400757
list index out of range
Met 4 F1 Score: 0.6774038076400757
list index out of range
Unmet 5 F1 Score: 0.6774038076400757
Unmet 6 F1 Score: 0.7035634517669678
Unmet 7 F1 Score: 0.8406149744987488
Unmet 8 F1 Score: 0.7359622716903687
Unmet 9 F1 Score: 0.5981931686401367


In [ ]:
# load scores from pickle file - gpt4
with open(gpt4_save_bert, 'rb') as f:
    loaded_scores = pickle.load(f)
print(loaded_scores)

loaded_scores = pd.DataFrame(loaded_scores, index=['values'])
loaded_scores = loaded_scores.transpose()
print(f"Average BERTScore Met Notes GPT4: {float((((loaded_scores.iloc[:5])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average BERTScore Unmet Notes GPT4: {float((((loaded_scores.iloc[5:])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average BERTScore All Notes GPT4: {float((((loaded_scores)['values']).sum()) / (len((loaded_scores)['values'])))}")



In [ ]:
bertScore(gpt3_load, gpt3_save_bert)

In [ ]:
# load scores from pickle file - gpt3
with open(gpt3_save_bert, 'rb') as f:
    loaded_scores = pickle.load(f)
print(loaded_scores)
loaded_scores = pd.DataFrame(loaded_scores, index=['values'])
loaded_scores = loaded_scores.transpose()

print(f"Average BERTScore Met Notes GPT3: {float((((loaded_scores.iloc[:5])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average BERTScore Unmet Notes GPT3: {float((((loaded_scores.iloc[5:])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average BERTScore All Notes GPT3: {float((((loaded_scores)['values']).sum()) / (len((loaded_scores)['values'])))}")


# Lexical Overlap Metric - METEOR

In [ ]:
def calculate_meteor(loadingFile, savingFile):
    # load all files from data folder (generated in Google Colab)
    met_dfs = {}
    unmet_dfs = {}
    for file in os.listdir(loadingFile):
        if '.csv' in file:
            
            if 'unmet' in file:
                unmet_dfs[file.replace('.csv', '')] = pd.read_csv(f'{loadingFile}{file}')
            else:
                met_dfs[file.replace('.csv', '')] = pd.read_csv(f'{loadingFile}{file}')
    nurse_notes = pd.read_excel('../../fake_notes.xlsx')

    score_dictionary = {}
    for i in range(len(nurse_notes)):
        meteor_scores = []
        word = ""
        if i < 5:
            candidates =  [re.sub(r'[^\w\s/]', '', i) for i in met_dfs[f'{i % (len(nurse_notes) // 2)}_met']['report']]
            word = "Met"
        else:
            candidates =  [re.sub(r'[^\w\s/]', '', i) for i in unmet_dfs[f'{i % (len(nurse_notes) // 2)}_unmet']['report']]
            word = "Unmet"
        reference = re.sub(r'[^\w\s/]', '', nurse_notes['Note'][i])
        for candidate in candidates:
            output_score = meteor([word_tokenize(candidate)], word_tokenize(reference))
            meteor_scores.append(output_score)
        score_dictionary[f'{i % (len(nurse_notes) // 2)}_{word}'] = (sum(meteor_scores)) / (len(meteor_scores))
    print(score_dictionary)

    # put scores in a pickle file
    with open(savingFile, 'wb') as f:
        pickle.dump(score_dictionary, f)

In [ ]:
calculate_meteor(gpt3_load, gpt3_save_meteor)

In [ ]:
# load scores from pickle file - gpt3
with open(gpt3_save_meteor, 'rb') as f:
    loaded_scores = pickle.load(f)
print(loaded_scores)
loaded_scores = pd.DataFrame(loaded_scores, index=['values'])
loaded_scores = loaded_scores.transpose()

print(f"Average METEOR Score Met Notes GPT3: {float((((loaded_scores.iloc[:5])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average METEOR Score Unmet Notes GPT3: {float((((loaded_scores.iloc[5:])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average METEOR Score All Notes GPT3: {float((((loaded_scores)['values']).sum()) / (len((loaded_scores)['values'])))}")

In [ ]:
calculate_meteor(gpt4_load, gpt4_save_meteor)

In [ ]:
# load scores from pickle file - gpt4
with open(gpt4_save_meteor, 'rb') as f:
    loaded_scores = pickle.load(f)
print(loaded_scores)
loaded_scores = pd.DataFrame(loaded_scores, index=['values'])
loaded_scores = loaded_scores.transpose()

print(f"Average METEOR Score Met Notes GPT4: {float((((loaded_scores.iloc[:5])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average METEOR Score Unmet Notes GPT4: {float((((loaded_scores.iloc[5:])['values']).sum()) / (len((loaded_scores.iloc[:5])['values'])))}")
print(f"Average METEOR Score All Notes GPT4: {float((((loaded_scores)['values']).sum()) / (len((loaded_scores)['values'])))}")

# Sentiment Analysis - TextBlob

In [ ]:
def preprocess_text(text):
    text = re.sub(r'[^\w\s/]', '', text)
    # tokenize
    tokens = word_tokenize(text.lower())
    # remove stop words
    filtered_tokens = [token for token in tokens if token not in stopwords.words('english')]
    # lemmatize the tokens
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
    # join the tokens back into a string
    processed_text = ' '.join(lemmatized_tokens)
    return processed_text

In [ ]:
def sentiment_interpreter(sentiment):
    sentiment = round(sentiment, 2)
    if sentiment > 0.5:
        return "positive"
    elif sentiment < -0.5: 
        return "negative"
    else:
        return "neutral"

In [ ]:
def calculate_sentiment(loadingFile, savingFile):
    # load all files from data folder (generated in Google Colab)
    met_dfs = {}
    unmet_dfs = {}
    for file in os.listdir(loadingFile):
        if '.csv' in file:
            
            if 'unmet' in file:
                unmet_dfs[file.replace('.csv', '')] = pd.read_csv(f'{loadingFile}{file}')
            else:
                met_dfs[file.replace('.csv', '')] = pd.read_csv(f'{loadingFile}{file}')
    nurse_notes = pd.read_excel('../../fake_notes.xlsx')

    score_dictionary = {}
    for i in range(len(nurse_notes)):
        nurse_note = preprocess_text(nurse_notes['Note'][i])
        nurse_blob = TextBlob(nurse_note)
        nurse_sentiment = nurse_blob.sentences[0].sentiment.polarity
        nurse_subjectivity = nurse_blob.sentences[0].sentiment.subjectivity
        word = ""
        if i < 5:
            candidates = [(TextBlob(preprocess_text(i))).sentences[0].sentiment.polarity for i in met_dfs[f'{i % (len(nurse_notes) // 2)}_met']['report']]
            generated_sentiment = (sum(candidates))/ len(candidates)
            candidates = [(TextBlob(preprocess_text(i))).sentences[0].sentiment.subjectivity for i in met_dfs[f'{i % (len(nurse_notes) // 2)}_met']['report']]
            generated_subjectivity = (sum(candidates))/ len(candidates)
            word = "Met"
        else:
            candidates = [(TextBlob(preprocess_text(i))).sentences[0].sentiment.polarity for i in unmet_dfs[f'{i % (len(nurse_notes) // 2)}_unmet']['report']]
            generated_sentiment = (sum(candidates))/ len(candidates)
            candidates = [(TextBlob(preprocess_text(i))).sentences[0].sentiment.subjectivity for i in unmet_dfs[f'{i % (len(nurse_notes) // 2)}_unmet']['report']]
            generated_subjectivity = (sum(candidates))/ len(candidates)
            word = "Unmet"
        score_dictionary[f'{i % (len(nurse_notes) // 2)}_{word}'] = {"Nurse Sentiment": nurse_sentiment, "Generated Sentiment": generated_sentiment, "Absolute Difference in Sentiment": math.sqrt((nurse_sentiment-generated_sentiment) ** 2), "Nurse Subjectivity": nurse_subjectivity , "Generated Subjectivity": generated_subjectivity, "Absolute Difference in Subjectivity":math.sqrt((nurse_subjectivity-generated_subjectivity) ** 2)}
    print(score_dictionary)
    # put scores in a pickle file
    with open(savingFile, 'wb') as f:
        pickle.dump(score_dictionary, f)

In [ ]:
calculate_sentiment(gpt3_load, gpt3_save_sentiment)

In [ ]:
def sentiment_parser(loaded_scores):
    nurse_met_sentiment = 0
    generated_met_sentiment = 0
    nurse_met_subjectivity = 0
    generated_met_subjectivity = 0
    met_count = 0
    nurse_unmet_sentiment = 0
    generated_unmet_sentiment = 0
    nurse_unmet_subjectivity = 0
    generated_unmet_subjectivity = 0
    unmet_count = 0
    for score in loaded_scores:
        if 'Unmet' in score:
            nurse_unmet_sentiment += loaded_scores[score]['Nurse Sentiment']
            generated_unmet_sentiment += loaded_scores[score]['Generated Sentiment']
            nurse_unmet_subjectivity += loaded_scores[score]['Nurse Subjectivity']
            generated_unmet_subjectivity += loaded_scores[score]['Generated Subjectivity']
            unmet_count += 1

        elif 'Met' in score:
            # print(loaded_scores[score]['Generated Sentiment'])
            nurse_met_sentiment += loaded_scores[score]['Nurse Sentiment']
            generated_met_sentiment += loaded_scores[score]['Generated Sentiment']
            nurse_met_subjectivity += loaded_scores[score]['Nurse Subjectivity']
            generated_met_subjectivity += loaded_scores[score]['Generated Subjectivity']
            met_count += 1
    # met note scores
    print("\n------------\nMet Notes\n------------\n")
    print(f"Nurse Met Sentiment: {nurse_met_sentiment / met_count}")
    print(f"Generated Met Sentiment: {generated_met_sentiment / met_count}")
    print(f"Difference in Met Sentiment: {math.sqrt(((nurse_met_sentiment / met_count)-(generated_met_sentiment / met_count)) ** 2)}")
    print(f"Nurse Met Subjectivity: {nurse_met_subjectivity / met_count}")
    print(f"Generated Met Subjectivity: {generated_met_subjectivity / met_count}")
    print(f"Difference in Met Subjectivity: {math.sqrt(((nurse_met_subjectivity / met_count)-(generated_met_subjectivity / met_count)) ** 2)}")

    # unmet note scores
    print("\n------------\nUnmet Notes\n------------\n")
    print(f"Nurse Unmet Sentiment: {nurse_unmet_sentiment / unmet_count}")
    print(f"Generated Unmet Sentiment: {generated_unmet_sentiment / unmet_count}")
    print(f"Difference in Unmet Sentiment: {math.sqrt(((nurse_unmet_sentiment / unmet_count)-(generated_unmet_sentiment / unmet_count)) ** 2)}")
    print(f"Nurse Unmet Subjectivity: {nurse_unmet_subjectivity / unmet_count}")
    print(f"Generated Unmet Subjectivity: {generated_unmet_subjectivity / unmet_count}")
    print(f"Difference in Unmet Subjectivity: {math.sqrt(((nurse_unmet_subjectivity / unmet_count)-(generated_unmet_subjectivity / unmet_count)) ** 2)}")


In [ ]:
# load scores from pickle file - gpt4
with open(gpt3_save_sentiment, 'rb') as f:
    loaded_scores = pickle.load(f)
print("GPT3")
print(loaded_scores)
sentiment_parser(loaded_scores=loaded_scores)

In [ ]:
calculate_sentiment(gpt4_load, gpt4_save_sentiment)

In [ ]:
# load scores from pickle file - gpt4
with open(gpt4_save_sentiment, 'rb') as f:
    loaded_scores = pickle.load(f)
print("GPT4")
print(loaded_scores)
sentiment_parser(loaded_scores=loaded_scores)